In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output


plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取存在数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.ads_hot_communities_daily 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_top_communities(city_name):
    """
    获取指定城市关注度 TOP 20 的小区。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        community, 
        MAX(total_attention) as hotness
    FROM iceberg_catalog.ershoufang.ads_hot_communities_daily
    WHERE city = '{city_name}' 
      AND community IS NOT NULL 
      AND community != ''
      AND community != '未知'
    GROUP BY community
    ORDER BY hotness DESC
    LIMIT 20
    """
    try:
        df = pd.read_sql(query, conn)
        df = df.sort_values(by='hotness', ascending=True).reset_index(drop=True)
        return df
    finally:
        conn.close()

In [10]:
def plot_hotness_barh(city):
    """绘制水平渐变条形图"""
    df = fetch_top_communities(city)
    
    if df.empty:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 暂无有效的小区热度数据", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 颜色渐变映射逻辑 ---
    norm = mcolors.Normalize(vmin=df['hotness'].min(), vmax=df['hotness'].max())
    cmap = cm.get_cmap('YlOrRd') 
    colors = cmap(norm(df['hotness']))

    # --- 2. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(12, 10))
    
    y_pos = np.arange(len(df['community']))
    values = df['hotness']

    bars = ax.barh(y_pos, values, color=colors, edgecolor='#E5E7E9', height=0.6, alpha=0.9)

    # --- 3. 标注与修饰 ---
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(
            width + (values.max() * 0.01), 
            bar.get_y() + bar.get_height()/2, 
            f"{int(width):,} 人关注", 
            ha='left', va='center', fontsize=11, fontweight='bold', color='#2C3E50'
        )

    plt.title(f'[{city}] 热门小区关注度排行榜', fontsize=18, pad=20, fontweight='bold')
    plt.xlabel('累计关注人数', fontsize=12, labelpad=10)
    plt.ylabel('小区名称', fontsize=12)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df['community'], fontsize=12)
    ax.set_xlim(0, values.max() * 1.15)

    ax.grid(axis='x', linestyle='--', alpha=0.4)
    sns.despine(left=True, top=True, right=True)

    plt.tight_layout()
    plt.show()

In [11]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_hotness_barh(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_hotness_barh(default_city)

In [12]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=1, layout=Layout(width='250px'), options=('上海', '北京', '南京', '天津', '太原', …

Output()